В цьому домашньому завданні ми знову працюємо з даними з нашого змагання ["Bank Customer Churn Prediction (DLU Course)"](https://www.kaggle.com/t/7c080c5d8ec64364a93cf4e8f880b6a0).

Тут ми побудуємо рішення задачі класифікації з використанням алгоритмів бустингу: XGBoost та LightGBM, а також використаємо бібліотеку HyperOpt для оптимізації гіперпараметрів.

0. Зчитайте дані `train.csv` в змінну `raw_df` та скористайтесь наведеним кодом нижче аби розділити дані на трнувальні та валідаційні і розділити дані на ознаки з матириці Х та цільову змінну. Назви змінних `train_inputs, train_targets, train_inputs, train_targets` можна змінити на ті, які Вам зручно.

  Наведений скрипт - частина отриманого мною скрипта для обробки даних. Ми тут не викнуємо масштабування та обробку категоріальних змінних, бо хочемо це делегувати алгоритмам, які будемо використовувати. Якщо щось не розумієте в наведених скриптах, рекомендую розібратись: навичка читати код - важлива складова роботи в машинному навчанні.

In [ ]:
# %pip install xgboost lightgbm hyperopt

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from typing import Tuple, Dict, Any


def split_train_val(df: pd.DataFrame, target_col: str, test_size: float = 0.2, random_state: int = 42) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Split the dataframe into training and validation sets.

    Args:
        df (pd.DataFrame): The raw dataframe.
        target_col (str): The target column for stratification.
        test_size (float): The proportion of the dataset to include in the validation split.
        random_state (int): Random state for reproducibility.

    Returns:
        Tuple[pd.DataFrame, pd.DataFrame]: Training and validation dataframes.
    """
    train_df, val_df = train_test_split(df, test_size=test_size, random_state=random_state, stratify=df[target_col])
    return train_df, val_df


def separate_inputs_targets(df: pd.DataFrame, input_cols: list, target_col: str) -> Tuple[pd.DataFrame, pd.Series]:
    """
    Separate inputs and targets from the dataframe.

    Args:
        df (pd.DataFrame): The dataframe.
        input_cols (list): List of input columns.
        target_col (str): Target column.

    Returns:
        Tuple[pd.DataFrame, pd.Series]: DataFrame of inputs and Series of targets.
    """
    inputs = df[input_cols].copy()
    targets = df[target_col].copy()
    return inputs, targets

In [2]:
raw_df = pd.read_csv('train.csv')
raw_df.head()

,id,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,0,15779985.0,Nwankwo,678.0,France,Male,29.0,4.0,0.00,3.0,1.0,0.0,180626.36,0.0
1,1,15650086.0,Ch'in,687.0,France,Female,34.0,1.0,0.00,2.0,0.0,1.0,63736.17,0.0
2,2,15733602.0,Thompson,682.0,France,Female,52.0,6.0,0.00,3.0,0.0,0.0,179655.87,1.0
3,3,15645794.0,Macleod,753.0,Germany,Male,44.0,6.0,83347.25,2.0,1.0,0.0,161407.48,0.0
4,4,15633840.0,Hsia,544.0,Germany,Female,55.0,0.0,107747.57,1.0,1.0,0.0,176580.86,1.0


In [3]:
train_df, val_df = split_train_val(raw_df, target_col='Exited')

In [4]:
target_col = 'Exited'

input_cols = raw_df.columns.drop(
    ['id', 'CustomerId', 'Surname', target_col]
).tolist()

In [5]:
train_inputs, train_targets = separate_inputs_targets(
    train_df, input_cols, target_col
)

val_inputs, val_targets = separate_inputs_targets(
    val_df, input_cols, target_col
)

In [6]:
print('Train inputs:', train_inputs.shape)
print('Train targets:', train_targets.shape)
print('Validation inputs:', val_inputs.shape)
print('Validation targets:', val_targets.shape)

Train inputs: (12000, 10)
Train targets: (12000,)
Validation inputs: (3000, 10)
Validation targets: (3000,)


Дані успішно розділено на тренувальну та валідаційну вибірки у співвідношенні 80/20. Відокремлено 10 вхідних ознак та цільову змінну Exited. Масштабування та кодування категоріальних ознак не виконувались.

1. В тренувальному та валідаційному наборі перетворіть категоріальні ознаки на тип `category`. Можна це зробити двома способами:
 1. `df[col_name].astype('category')`, як було продемонстровано в лекції
 2. використовуючи метод `pd.Categorical(df[col_name])`

In [7]:
categorical_cols = train_inputs.select_dtypes(
    include=['object', 'string', 'category']
).columns.tolist()

print('Categorical columns:', categorical_cols)

for col in categorical_cols:
    train_inputs[col] = train_inputs[col].astype('category')
    val_inputs[col] = val_inputs[col].astype('category')

print(train_inputs.dtypes)
print(val_inputs.dtypes)

Categorical columns: ['Geography', 'Gender']
CreditScore         float64
Geography          category
Gender             category
Age                 float64
Tenure              float64
Balance             float64
NumOfProducts       float64
HasCrCard           float64
IsActiveMember      float64
EstimatedSalary     float64
dtype: object
CreditScore         float64
Geography          category
Gender             category
Age                 float64
Tenure              float64
Balance             float64
NumOfProducts       float64
HasCrCard           float64
IsActiveMember      float64
EstimatedSalary     float64
dtype: object


2. Навчіть на отриманих даних модель `XGBoostClassifier`. Параметри алгоритму встановіть на свій розсуд, ми далі будемо їх тюнити. Рекомендую тренувати не дуже складну модель.

  Опис всіх конфігураційних параметрів XGBoostClassifier - тут https://xgboost.readthedocs.io/en/stable/parameter.html#global-config

  **Важливо:** зробіть такі налаштування `XGBoostClassifier` аби він самостійно обробляв незаповнені значення в даних і обробляв категоріальні колонки.

  Можна також, якщо працюєте в Google Colab, увімкнути можливість використання GPU (`Runtime -> Change runtime type -> T4 GPU`) і встановити параметр `device='cuda'` в `XGBoostClassifier` для пришвидшення тренування бустинг моделі.
  
  Після тренування моделі
  1. Виміряйте точність з допомогою AUROC на тренувальному та валідаційному наборах.
  2. Зробіть висновок про отриману модель: вона хороша/погана, чи є high bias/high variance?
  3. Порівняйте якість цієї моделі з тою, що ви отрмали з використанням DecisionTrees раніше. Чи вийшло покращити якість?

In [10]:
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    enable_categorical=True,
    missing=np.nan,
    tree_method='hist',
    random_state=42
)

xgb_model.fit(train_inputs, train_targets)

train_preds = xgb_model.predict_proba(train_inputs)[:, 1]
val_preds = xgb_model.predict_proba(val_inputs)[:, 1]

train_auc = roc_auc_score(train_targets, train_preds)
val_auc = roc_auc_score(val_targets, val_preds)

print(f'Train AUROC: {train_auc:.4f}')
print(f'Validation AUROC: {val_auc:.4f}')

Train AUROC: 0.9432
Validation AUROC: 0.9380


Висновок: модель XGBoost показала хороші результати: Train AUROC - 0.9432, Validation AUROC - 0.9380. Різниця між показниками невелика, тому явних ознак недонавчання (high bias) та перенавчання (high variance) немає. Порівняно з Decision Tree, якість моделі покращилася.

3. Використовуючи бібліотеку `Hyperopt` і приклад пошуку гіперпараметрів для `XGBoostClassifier` з лекції знайдіть оптимальні значення гіперпараметрів `XGBoostClassifier` для нашої задачі. Задайте свою сітку гіперпараметрів виходячи з тих параметрів, які ви б хотіли перебрати. Поставте кількість раундів в підборі гіперпараметрів рівну **20**.

  **Увага!** Для того, аби скористатись hyperopt, нам треба задати функцію `objective`. В ній ми маємо задати loss - це може будь-яка метрика, але бажано використовувтаи ту, яка цільова в вашій задачі. Чим менший лосс - тим ліпша модель на думку hyperopt. Тож, тут нам треба задати loss - негативне значення AUROC. В лекції ми натомість використовували Accuracy.

  Після успішного завершення пошуку оптимальних гіперпараметрів
    - виведіть найкращі значення гіперпараметрів
    - створіть в окремій зміній `final_clf` модель `XGBoostClassifier` з найкращими гіперпараметрами
    - навчіть модель `final_clf`
    - оцініть якість моделі `final_clf` на тренувальній і валідаційній вибірках з допомогою AUROC.
    - зробіть висновок про якість моделі. Чи стала вона краще порівняно з попереднім пунктом (2) цього завдання?

In [13]:
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK, space_eval

In [14]:
space = {
    'n_estimators': hp.choice('n_estimators', [50, 100, 150, 200]),
    'max_depth': hp.choice('max_depth', [3, 4, 5, 6]),
    'learning_rate': hp.uniform('learning_rate', 0.01, 0.2),
    'subsample': hp.uniform('subsample', 0.7, 1.0),
    'colsample_bytree': hp.uniform('colsample_bytree', 0.7, 1.0)
}

In [15]:
def objective(params):
    model = XGBClassifier(
        **params,
        enable_categorical=True,
        missing=np.nan,
        tree_method='hist',
        random_state=42
    )

    model.fit(train_inputs, train_targets)

    val_preds = model.predict_proba(val_inputs)[:, 1]
    auc = roc_auc_score(val_targets, val_preds)

    return {'loss': -auc, 'status': STATUS_OK}

In [16]:
trials = Trials()

best = fmin(
    fn=objective,
    space=space,
    algo=tpe.suggest,
    max_evals=20,
    trials=trials,
    rstate=np.random.default_rng(42)
)

best_params = space_eval(space, best)

print('Best parameters:', best_params)

100%|██████████| 20/20 [00:09<00:00,  2.22trial/s, best loss: -0.9378475889978738]
Best parameters: {'colsample_bytree': 0.9169443773332686, 'learning_rate': 0.09396924568696613, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.9264182080794368}


In [17]:
final_clf = XGBClassifier(
    **best_params,
    enable_categorical=True,
    missing=np.nan,
    tree_method='hist',
    random_state=42
)

final_clf.fit(train_inputs, train_targets)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.9169443773332686
,device,None
,early_stopping_rounds,None
,enable_categorical,True
,eval_metric,None


In [18]:
train_preds = final_clf.predict_proba(train_inputs)[:, 1]
val_preds = final_clf.predict_proba(val_inputs)[:, 1]

final_train_auc = roc_auc_score(train_targets, train_preds)
final_val_auc = roc_auc_score(val_targets, val_preds)

print(f'Train AUROC: {final_train_auc:.4f}')
print(f'Validation AUROC: {final_val_auc:.4f}')

Train AUROC: 0.9432
Validation AUROC: 0.9378


Висновок: після оптимізації гіперпараметрів за допомогою Hyperopt якість моделі практично не змінилася. Явних ознак недонавчання та перенавчання немає. Початкова модель XGBoost показала трохи кращий результат, тому доцільно залишити саме її.

4. Навчіть на наших даних модель LightGBM. Параметри алгоритму встановіть на свій розсуд, ми далі будемо їх тюнити. Рекомендую тренувати не дуже складну модель.

  Опис всіх конфігураційних параметрів LightGBM - тут https://lightgbm.readthedocs.io/en/latest/Parameters.html

  **Важливо:** зробіть такі налаштування LightGBM аби він самостійно обробляв незаповнені значення в даних і обробляв категоріальні колонки.

  Аби передати категоріальні колонки в LightGBM - необхідно виявити їх індекси і передати в параметрі `cat_feature=cat_feature_indexes`

  Після тренування моделі
  1. Виміряйте точність з допомогою AUROC на тренувальному та валідаційному наборах.
  2. Зробіть висновок про отриману модель: вона хороша/погана, чи є high bias/high variance?
  3. Порівняйте якість цієї моделі з тою, що ви отрмали з використанням XGBoostClassifier раніше. Чи вийшло покращити якість?

In [21]:
from lightgbm import LGBMClassifier

cat_feature_indexes = [
    train_inputs.columns.get_loc(col)
    for col in categorical_cols
]

print(cat_feature_indexes)

[1, 2]


In [22]:
lgbm_model = LGBMClassifier(
    n_estimators=100,
    max_depth=3,
    num_leaves=8,
    learning_rate=0.1,
    use_missing=True,
    random_state=42,
    verbosity=-1
)

lgbm_model.fit(
    train_inputs,
    train_targets,
    categorical_feature=cat_feature_indexes
)

,boosting_type,'gbdt'
,num_leaves,8
,max_depth,3
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [23]:
lgbm_train_preds = lgbm_model.predict_proba(train_inputs)[:, 1]
lgbm_val_preds = lgbm_model.predict_proba(val_inputs)[:, 1]

lgbm_train_auc = roc_auc_score(train_targets, lgbm_train_preds)
lgbm_val_auc = roc_auc_score(val_targets, lgbm_val_preds)

print(f'Train AUROC: {lgbm_train_auc:.4f}')
print(f'Validation AUROC: {lgbm_val_auc:.4f}')

Train AUROC: 0.9437
Validation AUROC: 0.9377


Висновок: модель LightGBM показала хороші результати. Різниця між Train AUROC (0.9437) та Validation AUROC (0.9377) невелика, тому явних ознак high bias та high variance немає. Порівняно з XGBoost, покращити якість не вдалося, проте результати моделей майже однакові.

5. Використовуючи бібліотеку `Hyperopt` і приклад пошуку гіперпараметрів для `LightGBM` з лекції знайдіть оптимальні значення гіперпараметрів `LightGBM` для нашої задачі. Задайте свою сітку гіперпараметрів виходячи з тих параметрів, які ви б хотіли перебрати. Поставте кількість раундів в підборі гіперпараметрів рівну **10**.

  **Увага!** Для того, аби скористатись hyperopt, нам треба задати функцію `objective`. І тут ми також ставимо loss - негативне значення AUROC, як і при пошуці гіперпараметрів для XGBoost. До речі, можна спробувати написати код так, аби в objective передавати лише модель і не писати схожий код двічі :)

  Після успішного завершення пошуку оптимальних гіперпараметрів
    - виведіть найкращі значення гіперпараметрів
    - створіть в окремій зміній `final_lgb_clf` модель `LightGBM` з найкращими гіперпараметрами
    - навчіть модель `final_lgb_clf`
    - оцініть якість моделі `final_lgb_clf` на тренувальній і валідаційній вибірках з допомогою AUROC.
    - зробіть висновок про якість моделі. Чи стала вона краще порівняно з попереднім пунктом (4) цього завдання?

In [24]:
lgb_space = {
    'n_estimators': hp.choice('n_estimators', [50, 100, 150, 200]),
    'max_depth': hp.choice('max_depth', [3, 4, 5, 6]),
    'num_leaves': hp.choice('num_leaves', [8, 16, 31]),
    'learning_rate': hp.uniform('learning_rate', 0.01, 0.2),
    'colsample_bytree': hp.uniform('colsample_bytree', 0.7, 1.0)
}

In [25]:
def lgb_objective(params):
    model = LGBMClassifier(
        **params,
        use_missing=True,
        random_state=42,
        verbosity=-1
    )

    model.fit(
        train_inputs,
        train_targets,
        categorical_feature=cat_feature_indexes
    )

    val_preds = model.predict_proba(val_inputs)[:, 1]
    auc = roc_auc_score(val_targets, val_preds)

    return {'loss': -auc, 'status': STATUS_OK}

In [26]:
lgb_trials = Trials()

lgb_best = fmin(
    fn=lgb_objective,
    space=lgb_space,
    algo=tpe.suggest,
    max_evals=10,
    trials=lgb_trials,
    rstate=np.random.default_rng(42)
)

lgb_best_params = space_eval(lgb_space, lgb_best)

print('Best parameters:', lgb_best_params)

100%|██████████| 10/10 [00:01<00:00,  5.85trial/s, best loss: -0.9375303518759861]
Best parameters: {'colsample_bytree': 0.911338004882514, 'learning_rate': 0.12256614853190341, 'max_depth': 3, 'n_estimators': 100, 'num_leaves': 16}


In [27]:
final_lgb_clf = LGBMClassifier(
    **lgb_best_params,
    use_missing=True,
    random_state=42,
    verbosity=-1
)

final_lgb_clf.fit(
    train_inputs,
    train_targets,
    categorical_feature=cat_feature_indexes
)

,boosting_type,'gbdt'
,num_leaves,16
,max_depth,3
,learning_rate,0.12256614853190341
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [28]:
final_lgb_train_preds = final_lgb_clf.predict_proba(train_inputs)[:, 1]
final_lgb_val_preds = final_lgb_clf.predict_proba(val_inputs)[:, 1]

final_lgb_train_auc = roc_auc_score(train_targets, final_lgb_train_preds)
final_lgb_val_auc = roc_auc_score(val_targets, final_lgb_val_preds)

print(f'Train AUROC: {final_lgb_train_auc:.4f}')
print(f'Validation AUROC: {final_lgb_val_auc:.4f}')

Train AUROC: 0.9453
Validation AUROC: 0.9375


Висновок: після оптимізації гіперпараметрів LightGBM якість на тренувальній вибірці трохи покращилася, а на валідаційній - незначно погіршилася. Явних ознак high bias та high variance немає. Оптимізація не покращила якість моделі на валідаційній вибірці, тому краще залишити початкову модель LightGBM.

6. Оберіть модель з експериментів в цьому ДЗ і зробіть новий `submission` на Kaggle та додайте код для цього і скріншот скора на публічному лідерборді.
  
  **Напишіть коментар, чому ви обрали саме цю модель?**

  І я вас вітаю - це останнє завдання з цим набором даних 💪 На цьому етапі корисно проаналізувати, які моделі показали себе найкраще і подумати, чому.